# 02 - Data Cleaning and Feature Preparation

This notebook documents the leakage decision for `Monthly_Salary`, builds the full and conservative feature sets, uses median/mode imputation in the preprocessing pipeline, and creates department profile tables for the counterfactual assignment step.

In [ ]:
from pathlib import Path
import json
import sys

import pandas as pd
from IPython.display import Markdown, display

current = Path.cwd().resolve()
candidate_roots = [current, current.parent, current.parent.parent]
ROOT = None
for candidate in candidate_roots:
    if (candidate / 'src' / 'smartassign_pipeline.py').exists():
        ROOT = candidate
        break
if ROOT is None:
    raise FileNotFoundError('Could not locate the repository root.')

sys.path.insert(0, str(ROOT / 'src'))
from smartassign_pipeline import (
    PROCESSED_DATA_DIR,
    RESULTS_DIR,
    basic_data_overview,
    compute_department_profiles,
    ensure_output_directories,
    get_counterfactual_profile_columns,
    get_feature_sets,
    load_raw_data,
)

ensure_output_directories()
pd.set_option('display.max_columns', None)

df = load_raw_data()
overview = basic_data_overview(df)
feature_sets = get_feature_sets(df)

print(f'Shape: {overview["shape"]}')
print('Total missing values:', int(df.isna().sum().sum()))
print('Missing-value handling: median imputation for numeric columns and most-frequent imputation for categorical columns inside the model pipeline.')

cleaned_df = df.copy()
cleaned_df['Hire_Date'] = pd.to_datetime(cleaned_df['Hire_Date'], errors='coerce')
cleaned_df.to_csv(PROCESSED_DATA_DIR / 'cleaned_employee_data.csv', index=False)

print('Full feature set:')
print(feature_sets['full'])
print('\nConservative feature set:')
print(feature_sets['conservative'])

for feature_set_name, feature_columns in feature_sets.items():
    profile_columns = get_counterfactual_profile_columns(feature_set_name, df)
    department_profiles = compute_department_profiles(df, profile_columns)
    profile_path = PROCESSED_DATA_DIR / f'department_profiles_{feature_set_name}.csv'
    department_profiles.to_csv(profile_path, index=False)
    print(f'\nDepartment profiles for {feature_set_name}:')
    display(department_profiles)
    print(f'Counterfactual profile columns used for {feature_set_name}: {profile_columns}')

metadata = {
    'leakage_feature_removed': 'Monthly_Salary',
    'excluded_columns_from_modeling': ['Employee_ID', 'Monthly_Salary', 'Performance_Score', 'Resigned', 'Hire_Date'],
    'encoding_choice': 'One-hot encoding for Department, Job_Title, Gender, and Education_Level because all have low cardinality in this dataset.',
    'train_test_split': '80/20 split stratified by Department.',
}
(RESULTS_DIR / 'data_prep_metadata.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')

summary_lines = [
    'Missing values are handled in the preprocessing pipeline with median imputation for numeric columns and most-frequent imputation for categorical columns.',
    'Monthly_Salary is removed from all modeling features because it is a documented leakage variable correlated with Performance_Score.',
    'Two feature sets are created: a full set that keeps Projects_Handled, Promotions, and Sick_Days, and a conservative set that drops them.',
    'Department and Job_Title are encoded with one-hot encoding because their cardinality is low in this dataset.',
    'Department profile medians are saved for the counterfactual assignment step, and the selection-bias assumption is documented explicitly.',
]
display(Markdown('### Data Prep Summary\n' + '\n'.join(f'- {line}' for line in summary_lines)))